### **Pandas + MSSQL Roadmap**

1. **Install Required Libraries**
```python
pip install pandas sqlalchemy pyodbc
```
#### **Connect to SQL Server**

In [1]:
import pandas as pd                           # Pandas library (used for read_sql() and to_sql())
from sqlalchemy import create_engine          # Creates the database connection (Engine)
from urllib.parse import quote_plus           # Encodes the connection string into a URL-safe format

# ODBC connection string
connection_string = (
    "DRIVER={ODBC Driver 17 for SQL Server};"   # Installed ODBC driver
    "SERVER=MUNNA\\SQLEXPRESS;"                 # SQL Server instance name
    "DATABASE=SalesDB;"                         # Database to connect to
    "Trusted_Connection=yes;"                   # Use Windows Authentication
)
# Windows Authentication → Uses your Windows login (Trusted_Connection=yes).
# SQL Server Authentication → Uses a SQL Server username and password (UID and PWD).

# Create a SQLAlchemy Engine (connection object)
engine = create_engine(
    # mssql  -> Microsoft SQL Server
    # pyodbc -> Use the pyodbc driver to communicate with SQL Server
    # odbc_connect -> Pass the complete ODBC connection string
    # quote_plus() -> URL-encode the connection string
    f"mssql+pyodbc:///?odbc_connect={quote_plus(connection_string)}"
)

# 'engine' is now used with Pandas:
# pd.read_sql("SELECT * FROM Employees", engine)
# df.to_sql("Employees", engine, if_exists="append", index=False)

#### 1. **Read Data (pd.read_sql())**

In [2]:
df = pd.read_sql(
    "SELECT * FROM Sales.Customers",
    engine)
df

,CustomerID,FirstName,LastName,Country,Score
0,1,Jossef,Goldberg,Germany,350
1,2,Kevin,Brown,USA,900
2,3,Mary,NaN,USA,750
3,4,Mark,Schwarz,Germany,500
4,5,Anna,Adams,USA,0


In [3]:
query ="""
SELECT *
FROM Sales.Employees
"""
df2 = pd.read_sql(query, engine)
df2

,EmployeeID,FirstName,LastName,Department,BirthDate,Gender,Salary,ManagerID
0,1,Frank,Lee,Marketing,1988-12-05,M,55000,NaN
1,2,Kevin,Brown,Marketing,1972-11-25,M,65000,1.0
2,3,Mary,NaN,Sales,1986-01-05,F,75000,1.0
3,4,Michael,Ray,Sales,1977-02-10,M,90000,2.0
4,5,Carol,Baker,Sales,1982-02-11,F,55000,3.0
5,6,Maria,Doe,HR,1988-01-12,F,80000,3.0


#### 2. **df.to_sql()**

Saves a DataFrame to a SQL database table.

**Syntax:**
```python
df.to_sql(
    name,
    con,
    if_exists='fail',
    index=True
)
```

**Important Parameters:**

<table style="margin-left: 0; margin-right: auto;">
<tr>
<th>Parameter</th>
<th>Meaning</th>
<th>Default</th>
</tr>
<tr>
<td><i>name</i></td>
<td>Name of the SQL table</td>
<td>Required</td>
</tr>
<tr>
<td><i>con</i></td>
<td>Database connection or engine</td>
<td>Required</td>
</tr>
<tr>
<td><i>if_exists</i></td>
<td>What to do if the table already exists</td>
<td>'fail'</td>
</tr>
<tr>
<td><i>index</i></td>
<td>Save the DataFrame index as a column</td>
<td>True</td>
</tr>
</table>

**if_exists values**

<table style="margin-left: 0; margin-right: auto;">
<tr>
<th>Value</th>
<th>Meaning</th>
</tr>
<tr>
<td><i>'fail'</i></td>
<td>Raise an error if the table exists</td>
</tr>
<tr>
<td><i>'replace'</i></td>
<td>Drop the existing table and create a new one</td>
</tr>
<tr>
<td><i>'append'</i></td>
<td>Add rows to the existing table</td>
</tr>
</table>

In [4]:
df = pd.read_csv(r'sample_datasets\Students.csv')
df.head()

,Student_ID,Name,Gender,Department,Year,Math,Science,English,City
0,1001,Krishna,M,CSE,2,67,67,92,Vijayawada
1,1002,Meera,F,MECH,1,43,67,54,Vijayawada
2,1003,Keerthi,F,CSE,4,77,95,95,Pune
3,1004,Priya,F,CSE,4,82,85,73,Chennai
4,1005,Rahul,M,ECE,4,73,77,35,Vijayawada


In [5]:
df.to_sql(
    'Students',
    con=engine,
    if_exists='replace',
    index=False
)

100

In [6]:
pd.read_sql("SELECT * FROM Students", engine)

,Student_ID,Name,Gender,Department,Year,Math,Science,English,City
0,1001,Krishna,M,CSE,2,67,67,92,Vijayawada
1,1002,Meera,F,MECH,1,43,67,54,Vijayawada
2,1003,Keerthi,F,CSE,4,77,95,95,Pune
3,1004,Priya,F,CSE,4,82,85,73,Chennai
4,1005,Rahul,M,ECE,4,73,77,35,Vijayawada
...,...,...,...,...,...,...,...,...,...
95,1096,Sneha,F,CSE,3,66,73,73,Chennai
96,1097,Vivaan,M,MECH,2,58,53,60,Mumbai
97,1098,Aarav,M,CSE,2,46,73,68,Vijayawada
98,1099,Divya,F,MECH,2,84,79,88,Pune


In [7]:
pd.read_sql("SELECT * FROM Sales.Products", engine)

,ProductID,Product,Category,Price
0,101,Bottle,Accessories,10
1,102,Tire,Accessories,15
2,103,Socks,Clothing,20
3,104,Caps,Clothing,25
4,105,Gloves,Clothing,30
